# 86 — Campaign C q&lt;1 hunt → Campaign B handoff

1. 現検索を診断（staged screening q&lt;1 があるか）
2. 初回のみ拡張 Campaign C（`qlt1c`）を一度実行
3. **それでも q≥1 なら Campaign B（S2）へ**（同じ C を繰り返さない）
4. staged q&lt;1 が出たら notebook **85**

- screening q&lt;1 は証明書ではない（`NOT_CERTIFIED`）
- production M6 禁止


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'm7_q_lt1_hunt.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/m7_q_lt1_hunt.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
from src.m7_q_lt1_hunt import diagnose_q_lt1_hunt, next_q_lt1_actions
from src.m7_status import M7_RUN_ID_CAMPAIGN_C
from src.runtime import validate_persistent_root

validate_persistent_root()
CURRENT_M7C = os.environ.get('VALIDATED_RG_M7C_RUN_ID', M7_RUN_ID_CAMPAIGN_C)
DIAG = diagnose_q_lt1_hunt(
    persistent_root=PERSIST_ROOT,
    search_run_id=CURRENT_M7C,
)
ACTIONS = next_q_lt1_actions(DIAG)
print(json.dumps({
    'search_run_id': DIAG['search_run_id'],
    'counts': DIAG['counts'],
    'best_staged': DIAG['best_staged'],
    'recommendation': DIAG['recommendation'],
    'note': DIAG['note'],
    'next_action': ACTIONS,
}, indent=2, ensure_ascii=False))


## 拡張 Campaign C 検索（staged q&lt;1 が無いときだけ）

新しい run id を発行し、coupling / seed を広げた空間で再検索する。


In [ ]:
from src.m7_config import default_m7_config
from src.m7_orchestrator import create_or_resume_m7
from src.m7_status import M6_PARENT_RUN_ID_FROZEN
from src.m7_q_lt1_hunt import mint_m7c_qlt1_run_id

if ACTIONS.get('action') != 'notebook_86_new_search':
    print('Skip new search:', ACTIONS.get('action'), ACTIONS.get('note'))
    M7C_NEW_ID = None
    M7C_RESULT = None
else:
    M7C_NEW_ID = ACTIONS.get('suggested_run_id') or mint_m7c_qlt1_run_id()
    os.environ['VALIDATED_RG_M7C_RUN_ID'] = M7C_NEW_ID
    print('NEW M7C_RUN_ID', M7C_NEW_ID)
    config_c = default_m7_config(
        parent_m6_run_id=os.environ.get('VALIDATED_RG_M6_RUN_ID', M6_PARENT_RUN_ID_FROZEN),
        run_id=M7C_NEW_ID,
        mode='paperspace',
        campaign='C',
        lineage_mode='auto',
        human_review_approved=True,
        auto_approve_for_materialize=False,
        max_executable_j2_max=2,
        max_staged_j2_max=2,
        max_candidates_total=96,
        max_rigorous_replays=24,
        max_lineage_replays=2,
        stop_on_first_certified=True,
    )
    m7c = create_or_resume_m7(PERSIST_ROOT, config_c, PROJECT_ROOT)
    M7C_RESULT = m7c.run_search()
    print(json.dumps({
        k: M7C_RESULT.get(k)
        for k in ('status', 'phase', 'run_id', 'message', 'note')
        if k in (M7C_RESULT or {})
    }, indent=2, ensure_ascii=False, default=str))


In [ ]:
HUNT_ID = os.environ.get('VALIDATED_RG_M7C_RUN_ID', CURRENT_M7C)
DIAG2 = diagnose_q_lt1_hunt(persistent_root=PERSIST_ROOT, search_run_id=HUNT_ID)
ACTIONS2 = next_q_lt1_actions(DIAG2)
print(json.dumps({
    'search_run_id': DIAG2['search_run_id'],
    'counts': DIAG2['counts'],
    'best_staged': DIAG2['best_staged'],
    'staged_q_lt1_live': DIAG2['staged_q_lt1_live'],
    'recommendation': DIAG2['recommendation'],
    'note': DIAG2['note'],
    'next_action': ACTIONS2.get('action'),
}, indent=2, ensure_ascii=False))
if ACTIONS2.get('action') == 'notebook_85':
    print('NEXT: notebook 85 with:')
    print(f"os.environ['VALIDATED_RG_M7C_RUN_ID'] = {HUNT_ID!r}")
elif ACTIONS2.get('action') == 'notebook_86_campaign_b':
    print('NEXT: run Campaign B cell below (do NOT mint another Campaign C).')
    print(ACTIONS2.get('note'))
else:
    print('NEXT action', ACTIONS2.get('action'))
    print(ACTIONS2.get('note'))


## Campaign B — S2 rank/residual（C が q≥1 で尽きたとき）

診断が `CAMPAIGN_C_S3_EXHAUSTED_GOTO_B` / `notebook_86_campaign_b` のときだけ実行。


In [ ]:
from src.m7_config import default_m7_config
from src.m7_orchestrator import create_or_resume_m7
from src.m7_status import M6_PARENT_RUN_ID_FROZEN, M7_RUN_ID_CAMPAIGN_B

action = (ACTIONS2 if 'ACTIONS2' in globals() else ACTIONS).get('action')
if action != 'notebook_86_campaign_b':
    print('Skip Campaign B:', action)
    M7B_RESULT = None
else:
    os.environ['VALIDATED_RG_M7B_RUN_ID'] = M7_RUN_ID_CAMPAIGN_B
    config_b = default_m7_config(
        parent_m6_run_id=os.environ.get('VALIDATED_RG_M6_RUN_ID', M6_PARENT_RUN_ID_FROZEN),
        run_id=M7_RUN_ID_CAMPAIGN_B,
        mode='paperspace',
        campaign='B',
        lineage_mode='auto',
        human_review_approved=True,
        auto_approve_for_materialize=False,
        max_candidates_total=48,
        max_rigorous_replays=24,
        max_lineage_replays=2,
        stop_on_first_certified=True,
    )
    m7b = create_or_resume_m7(PERSIST_ROOT, config_b, PROJECT_ROOT)
    M7B_RESULT = m7b.run_search()
    print(json.dumps(M7B_RESULT, indent=2, ensure_ascii=False, default=str)[:4000])
    print('Campaign B done. Inspect ranking/plans; screening q is NOT_CERTIFIED.')
